In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("All libraries loaded successfully")

All libraries loaded successfully


In [2]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

books = soup.find_all("article", class_="product_pod")
print(f"Found {len(books)} books on this page")

# Let's look at just the first one to understand the structure
first_book = books[0]
print(first_book.prettify()[:800])


Found 20 books on this page
<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." 


In [3]:
url = "https://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
response.encoding = "utf-8"   # <-- the fix for the £ symbol issue
soup = BeautifulSoup(response.text, "html.parser")

books = soup.find_all("article", class_="product_pod")

def extract_book(book):
    # Full title is in the <a> tag's "title" attribute (not the visible text, which gets cut off)
    title = book.h3.a["title"]

    # Price: strip the £ symbol and convert to a number
    price_text = book.find("p", class_="price_color").text
    price = float(price_text.replace("£", "").strip())

    # Rating: the word is the SECOND class on the <p> tag, e.g. class="star-rating Three"
    rating_classes = book.find("p", class_="star-rating")["class"]
    rating_word = rating_classes[1]  # ["star-rating", "Three"]

    # Stock status
    stock_text = book.find("p", class_="instock availability").text.strip()

    # Link to the book's own detail page (we'll use this later for quantity + genre)
    detail_url = book.h3.a["href"]

    return {
        "title": title,
        "price": price,
        "rating_word": rating_word,
        "stock_status": stock_text,
        "detail_url": detail_url
    }

# Test it on all 20 books on this page
page1_books = [extract_book(b) for b in books]

import pandas as pd
df_test = pd.DataFrame(page1_books)
df_test

,title,price,rating_word,stock_status,detail_url
0,A Light in the Attic,51.77,Three,In stock,a-light-in-the-attic_1000/index.html
1,Tipping the Velvet,53.74,One,In stock,tipping-the-velvet_999/index.html
2,Soumission,50.10,One,In stock,soumission_998/index.html
3,Sharp Objects,47.82,Four,In stock,sharp-objects_997/index.html
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,sapiens-a-brief-history-of-humankind_996/index...
5,The Requiem Red,22.65,One,In stock,the-requiem-red_995/index.html
6,The Dirty Little Secrets of Getting Your Dream...,33.34,Four,In stock,the-dirty-little-secrets-of-getting-your-dream...
7,The Coming Woman: A Novel Based on the Life of...,17.93,Three,In stock,the-coming-woman-a-novel-based-on-the-life-of-...
8,The Boys in the Boat: Nine Americans and Their...,22.60,Four,In stock,the-boys-in-the-boat-nine-americans-and-their-...
9,The Black Maria,52.15,One,In stock,the-black-maria_991/index.html
